In [1]:
from __future__ import annotations

import sys
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 240)
pd.set_option("display.max_rows", 300)

CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError(
        "Could not find project root. Run this notebook from the project root or notebooks/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

from src.io.database_h5 import load_nir_uco_h5

from src.utils import (
    save_parquet,
    save_parquet_if_nonempty,
    load_parquet,
    list_result_files,
    parse_preprocessing_steps,
)

from src.spectra.band_selection import (
    select_wavelength_range_from_database,
    wavelength_selection_summary,
)

from src.spectra.preprocessing_configs import normalize_preprocessing_configs

from src.decision.metrics import (
    add_binary_confusion_case,
)

from src.decision.border import (
    summarize_pixel_errors_by_border_zone,
    summarize_border_diagnostics_by_config,
)

from src.decision.uncertainty import (
    add_three_way_object_decision,
    evaluate_three_way_object_decision,
    apply_three_way_thresholds_by_config,
    evaluate_three_way_by_config,
)

from src.workflows.simca import (
    make_target_train_filters,
    refit_selected_simca_configs,
    run_selected_simca_random_state_stability_full,
    run_selected_simca_random_state_stability,
)

from src.workflows.simca_selection_utils import (
    ensure_candidate_columns,
    normalize_simca_rule_columns,
    add_detection_selection_score,
    sort_detection_selection,
    add_reference_selection_scores,
    fill_selected_config_defaults,
    summarize_parameter_tendencies,
    summarize_ablation_effects,
    summarize_metric_stability,
    pareto_front_by_group,
)

from src.visualization.plot_model_selection import (
    plot_detection_pareto,
    plot_three_way_tradeoff,
)

from src.visualization.plot_robustness import (
    plot_ablation_deltas,
    plot_stability_intervals,
    plot_border_core_metrics,
)

%load_ext autoreload
%autoreload 2

PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


In [2]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
DB_H5_PATH = (
    PROJECT_ROOT
    / "HSI Data"
    / "processed"
    / "nir_uco_database.h5"
)

# ---------------------------------------------------------------------
# Spectral configuration
# ---------------------------------------------------------------------
# Current workflow:
#   use all non-noisy bands stored in the H5 database.
#
# Later, set USE_WAVELENGTH_WINDOW=True to test a spectral window.
USE_WAVELENGTH_WINDOW = False
WAVELENGTH_MODE = "non_noisy_all"

WINDOW_MIN_NM = 1225.0
WINDOW_MAX_NM = 1675.0

if USE_WAVELENGTH_WINDOW:
    RESULTS_TAG = f"{int(WINDOW_MIN_NM)}_{int(WINDOW_MAX_NM)}"
else:
    RESULTS_TAG = "non_noisy_all"

# ---------------------------------------------------------------------
# Input from 04A
# ---------------------------------------------------------------------
RESULTS_04A_DIR = (
    PROJECT_ROOT
    / "results"
    / f"04A_simca_grid_validation_{RESULTS_TAG}"
)

GRID_SUMMARY_PATH = (
    RESULTS_04A_DIR
    / "grid_summary.parquet"
)

PARETO_2WAY_04A_PATH = (
    RESULTS_04A_DIR
    / "pareto_2way_candidates_large.parquet"
)

THREE_WAY_GRID_04A_PATH = (
    RESULTS_04A_DIR
    / "three_way_threshold_grid.parquet"
)

SELECTED_CANDIDATE_CONFIGS_PATH = (
    RESULTS_04A_DIR
    / "selected_candidate_configs.parquet"
)

# ---------------------------------------------------------------------
# Outputs
# ---------------------------------------------------------------------
RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / f"04B_simca_validation_robustness_{RESULTS_TAG}"
)
#DEBUG_DIR = RESULTS_DIR / "debug"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
#DEBUG_DIR.mkdir(parents=True, exist_ok=True)

ABLATION_SUMMARY_PATH = RESULTS_DIR / "ablation_summary.parquet"

VALIDATION_REFIT_METRICS_PATH = RESULTS_DIR / "validation_refit_metrics.parquet"
VALIDATION_PIXEL_ERRORS_BY_IMAGE_PATH = RESULTS_DIR / "validation_pixel_errors_by_image.parquet"
VALIDATION_REFIT_ERRORS_PATH = RESULTS_DIR / "validation_refit_errors.parquet"

RANDOM_STATE_STABILITY_PATH = RESULTS_DIR / "random_state_stability.parquet"
RANDOM_STATE_STABILITY_ERRORS_PATH = RESULTS_DIR / "random_state_stability_errors.parquet"

BORDER_DIAGNOSTIC_VALIDATION_PATH = RESULTS_DIR / "border_diagnostic_validation.parquet"

ROBUSTNESS_SUMMARY_PATH = RESULTS_DIR / "robustness_summary.parquet"
ROBUST_CANDIDATE_CONFIGS_PATH = RESULTS_DIR / "robust_candidate_configs.parquet"

VALIDATION_ROBUSTNESS_PROTOCOL_PATH = RESULTS_DIR / "validation_robustness_protocol.parquet"

VALIDATION_REFIT_OBJECTS_PATH = RESULTS_DIR / "validation_refit_objects.parquet"
VALIDATION_REFIT_PIXELS_PATH = RESULTS_DIR / "validation_refit_pixels.parquet"

VALIDATION_3WAY_OBJECTS_PATH = RESULTS_DIR / "validation_3way_objects.parquet"
VALIDATION_3WAY_METRICS_PATH = RESULTS_DIR / "validation_3way_metrics.parquet"

RANDOM_STATE_STABILITY_OBJECTS_PATH = RESULTS_DIR / "random_state_stability_objects.parquet"
RANDOM_STATE_STABILITY_3WAY_METRICS_PATH = RESULTS_DIR / "random_state_stability_3way_metrics.parquet"
RANDOM_STATE_STABILITY_3WAY_SUMMARY_PATH = RESULTS_DIR / "random_state_stability_3way_summary.parquet"

ROBUST_2WAY_PARETO_PATH = RESULTS_DIR / "robust_2way_pareto_candidates.parquet"
ROBUST_3WAY_PARETO_PATH = RESULTS_DIR / "robust_3way_pareto_candidates.parquet"
ROBUST_PARETO_UNION_PATH = RESULTS_DIR / "robust_pareto_union_candidates.parquet"

ROBUST_CANDIDATE_CONFIGS_PATH = RESULTS_DIR / "04B_final_candidate_panel.parquet"

# ---------------------------------------------------------------------
# Protocol
# ---------------------------------------------------------------------
TARGET_CLASS = "peanut"
NON_TARGET_LABEL = "almond"
REFERENCE_CLASSES = ("almond", TARGET_CLASS)

TRAIN_FILTERS = make_target_train_filters(
    target_class=TARGET_CLASS,
    train_batches=[1, 2],
)

VALIDATION_FILTERS = {
    "sample_kind": ["pure"],
    "object_nut_type": list(REFERENCE_CLASSES),
    "batch": [3],
}

# ---------------------------------------------------------------------
# Runtime
# ---------------------------------------------------------------------
RANDOM_STATE = 42
REPLACE_BALANCED_PIXELS = False

CV_N_SPLITS = 5
CV_GROUP_COL = "object_id"

# Random-state stability.
RUN_RANDOM_STATE_STABILITY = True
RANDOM_STABILITY_SEEDS = [0, 1, 2, 3, 4, 5, 10, 20, 42, 100]
# Usually only balanced_pixels needs seed stability.
RUN_STABILITY_ONLY_FOR_BALANCED_PIXELS = True

# Border diagnostic.
RUN_BORDER_DIAGNOSTIC = True
BORDER_DIAGNOSTIC_WIDTHS = [1, 2, 3]
BORDER_DIAGNOSTIC_CONFIG_LIMIT_PER_FAMILY = 30

# Robust filtering thresholds.
MAX_MEAN_FN_RATE = 0.20
MAX_STD_FN_RATE = 0.08
MAX_MAX_FN_RATE = 0.35
MAX_MEAN_FP_RATE = 0.60

# Do not exclude border-sensitive models automatically at first.
EXCLUDE_BORDER_SENSITIVE_MODELS = False
MAX_CORE_FN_RATE = 0.30

# Final robust panel.
N_ROBUST_PER_MATRIX_FAMILY = 20
N_ROBUST_OVERALL = 40

THREE_WAY_LOWER_THRESHOLDS = np.round(np.arange(0.05, 0.61, 0.05), 2)
THREE_WAY_UPPER_THRESHOLDS = np.round(np.arange(0.50, 0.96, 0.05), 2)
MAX_THREE_WAY_TARGET_MISS_RATE = 0.00
MAX_THREE_WAY_FALSE_ACCEPT_RATE = 0.30
MAX_THREE_WAY_UNCERTAIN_RATE = 0.60

# ---------------------------------------------------------------------
# Candidate panel size
# ---------------------------------------------------------------------
MAX_CANDIDATES_FOR_REFIT_PER_FAMILY = 60

# ---------------------------------------------------------------------
# Robust Pareto objectives
# ---------------------------------------------------------------------
ROBUST_2WAY_MINIMIZE_COLS = [
    "max_fn_rate",
    "mean_fn_rate",
    "std_fn_rate",
    "mean_fp_rate",
    "std_fp_rate",
]

ROBUST_2WAY_MAXIMIZE_COLS = [
    "mean_balanced_accuracy",
]

ROBUST_3WAY_MINIMIZE_COLS = [
    "threeway_max_target_miss_rate",
    "threeway_mean_target_miss_rate",
    "threeway_mean_non_target_false_accept_rate",
    "threeway_mean_uncertain_rate",
    "threeway_std_uncertain_rate",
]

ROBUST_3WAY_MAXIMIZE_COLS = [
    "threeway_mean_coverage_rate",
]

# Optional permissive filters before final panel.
MAX_ALLOWED_MAX_FN_RATE = 0.35
MAX_ALLOWED_MEAN_FN_RATE = 0.20
MAX_ALLOWED_MEAN_TARGET_MISS_RATE = 0.20
MAX_ALLOWED_MEAN_UNCERTAIN_RATE = 0.80

print("DB_H5_PATH:", DB_H5_PATH)
print("RESULTS_04A_DIR:", RESULTS_04A_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("SELECTED_CANDIDATE_CONFIGS_PATH:", SELECTED_CANDIDATE_CONFIGS_PATH)

DB_H5_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\HSI Data\processed\nir_uco_database.h5
RESULTS_04A_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_validation_non_noisy_all
RESULTS_DIR: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_validation_robustness_non_noisy_all
SELECTED_CANDIDATE_CONFIGS_PATH: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04A_simca_grid_validation_non_noisy_all\selected_candidate_configs.parquet


In [3]:
if not DB_H5_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_H5_PATH}. Run notebook 00 first.")

if not GRID_SUMMARY_PATH.exists():
    raise FileNotFoundError(f"Missing 04A output: {GRID_SUMMARY_PATH}")

if not SELECTED_CANDIDATE_CONFIGS_PATH.exists():
    raise FileNotFoundError(f"Missing 04A output: {SELECTED_CANDIDATE_CONFIGS_PATH}")

object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=True,
)

if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, wavelength_info = select_wavelength_range_from_database(
        object_db=object_db,
        image_db=image_db,
        min_nm=WINDOW_MIN_NM,
        max_nm=WINDOW_MAX_NM,
    )
    wavelength_selection_df = wavelength_selection_summary(wavelength_info)
else:
    first_obj = next(iter(object_db.values()))
    wavelengths = first_obj.get("wavelengths")
    wavelengths = np.asarray(wavelengths, dtype=float) if wavelengths is not None else None
    wavelength_selection_df = pd.DataFrame()

if wavelengths is None:
    raise RuntimeError("No wavelength axis found in object_db.")

grid_summary_df = load_parquet(GRID_SUMMARY_PATH)
selected_configs_df = load_parquet(SELECTED_CANDIDATE_CONFIGS_PATH)

grid_summary_df = normalize_simca_rule_columns(grid_summary_df)
grid_summary_df = add_detection_selection_score(grid_summary_df)
grid_summary_df = add_reference_selection_scores(grid_summary_df)

selected_configs_df = ensure_candidate_columns(selected_configs_df)
selected_configs_df = normalize_simca_rule_columns(selected_configs_df)
selected_configs_df = fill_selected_config_defaults(
    selected_configs_df,
    default_values={
        "target_class": TARGET_CLASS,
        "non_target_label": NON_TARGET_LABEL,
        "sg_window_length": 11,
        "sg_polyorder": 2,
        "position_dilation_radius": 3,
        "m": 40,
        "alpha": 0.05,
        "object_threshold": 0.75,
    },
)
selected_configs_df = add_detection_selection_score(selected_configs_df)
selected_configs_df = add_reference_selection_scores(selected_configs_df)

candidate_panel_df = (
    selected_configs_df
    .sort_values(
        ["matrix_family", "fn_rate", "fp_rate", "balanced_accuracy"],
        ascending=[True, True, True, False],
    )
    .groupby("matrix_family", group_keys=False, dropna=False)
    .head(MAX_CANDIDATES_FOR_REFIT_PER_FAMILY)
    .reset_index(drop=True)
)

print("Database loaded")
print("n objects:", len(object_db))
print("n images:", len(image_db))
print("n active bands:", len(wavelengths))
print("Combined grid:", grid_summary_df.shape)
print("Selected configs:", selected_configs_df.shape)
print("Candidate panel for robustness:", candidate_panel_df.shape)

display(candidate_panel_df.head())

Database loaded
n objects: 1262
n images: 48
n active bands: 63
Combined grid: (42120, 62)
Selected configs: (22, 72)
Candidate panel for robustness: (22, 72)


,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,object_threshold,selection_score,search_method,model_family,matrix_family,training_matrix_id,matrix_method,m,m_effective,balanced_pixel_strategy,balanced_pixel_strategy_effective,preprocessing,preprocessing_steps,rule_variant,rule,n_components,alpha,non_target_label,sg_window_length,sg_polyorder,position_dilation_radius,cv_n_splits,n_cv_observations,n_cv_groups,H_emp_cv,Q_emp_cv,simple_emp_cv,alternative_chi2_emp_cv,alternative_empHQ_emp_cv,data_driven_emp_cv,cv_target_rejection_rate,cv_target_acceptance_rate,cv_expected_rejection_rate,cv_abs_rejection_error,cv_rule_limit,rule_original,rule_variant_original,rule_token,selected_rule_name,rule_for_refit,limit_source,selection_split,selection_strategy,grid_config_id,kept_after_within_same_model_preprocessing,kept_after_within_same_family_components,kept_after_family_level_pareto,selected_config_id,three_way_lower_threshold,three_way_upper_threshold,val3_target_miss_rate,val3_false_accept_rate,val3_uncertain_rate,val3_coverage_rate,score_conservative_target,score_balanced_reference,score_specificity_control
0,peanut,almond,108,53,0,52,3,1.000000,0.054545,0.527273,0.518519,0.504762,0.670886,0.000000,0.945455,0.75,-0.901540,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_mean,object_mean,40,40,not_applicable,random,absorbance_sg_smooth,absorbance+sg_smooth,data_driven_emp_cv,data_driven,11,0.01,almond,11,2,3,5,98,98,66.668457,5.685526e-05,16.527746,17.928278,1.512749,1333.969360,0.010204,0.989796,0.01,0.000204,1333.969360,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,grid_001689,True,True,True,04A_object_matrix_0001,0.05,0.95,0.0,0.545455,0.435185,0.564815,-1.363636,1.566509,-2.672727
1,peanut,almond,108,52,1,43,12,0.981132,0.218182,0.599657,0.592593,0.547368,0.702703,0.018868,0.781818,0.70,-0.923510,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,40,40,not_applicable,random,absorbance_sg_d2,absorbance+sg_d2,data_driven_emp_cv,data_driven,3,0.01,almond,11,2,3,5,98,98,17.640045,2.147866e-09,4.319395,4.536808,1.380628,384.212128,0.010204,0.989796,0.01,0.000204,384.212128,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,grid_009680,True,True,True,04A_object_matrix_0002,0.55,0.95,0.0,0.218182,0.712963,0.287037,-1.341338,1.959548,-1.785249
2,peanut,almond,108,50,3,41,14,0.943396,0.254545,0.598971,0.592593,0.549451,0.694444,0.056604,0.745455,0.75,-1.264918,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,40,40,not_applicable,random,absorbance_sg_d2,absorbance+sg_d2,data_driven_emp_cv,data_driven,3,0.01,almond,11,2,3,5,98,98,17.640045,2.147866e-09,4.319395,4.536808,1.380628,384.212128,0.010204,0.989796,0.01,0.000204,384.212128,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,grid_011947,True,True,True,04A_object_matrix_0003,0.55,0.95,0.0,0.218182,0.712963,0.287037,-2.024014,1.872387,-1.755746
3,peanut,almond,108,49,4,37,18,0.924528,0.327273,0.625901,0.620370,0.569767,0.705036,0.075472,0.672727,0.70,-1.379785,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,40,40,not_applicable,random,absorbance_sg_d2,absorbance+sg_d2,simple_emp_cv,simple,3,0.01,almond,11,2,3,5,98,98,17.640045,2.147866e-09,4.319395,4.536808,1.380628,384.212128,0.010204,0.989796,0.01,0.000204,4.319395,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,grid_012764,True,True,True,04A_object_matrix_0004,0.55,0.95,0.0,0.109091,0.814815,0.185185,-2.228988,1.993780,-1.413722
4,peanut,almon

In [4]:
print("Selected configs by matrix family:")
display(selected_configs_df["matrix_family"].value_counts(dropna=False))

print("Candidate panel by matrix family:")
display(candidate_panel_df["matrix_family"].value_counts(dropna=False))

if candidate_panel_df["matrix_family"].nunique(dropna=False) < 2:
    print(
        "[WARNING] Only one matrix family is present in candidate_panel_df. "
        "Check whether 04A selected both object_matrix and pixel_matrix candidates."
    )

Selected configs by matrix family:


matrix_family
object_matrix    16
pixel_matrix      6
Name: count, dtype: int64

Candidate panel by matrix family:


matrix_family
object_matrix    16
pixel_matrix      6
Name: count, dtype: int64

In [5]:
required_3way_cols = [
    "three_way_lower_threshold",
    "three_way_upper_threshold",
]

missing_3way_cols = [
    col for col in required_3way_cols
    if col not in candidate_panel_df.columns
]

if missing_3way_cols:
    raise KeyError(
        "The 04A selected candidate file does not contain fixed 3-way thresholds. "
        f"Missing columns: {missing_3way_cols}. "
        "Re-run 04A after the 3-way calibration cell."
    )

In [6]:
PREPROCESSING_CONFIGS = {
    str(row["preprocessing"]): tuple(
        parse_preprocessing_steps(
            row["preprocessing_steps"]
        )
    )
    for _, row in (
        candidate_panel_df
        .drop_duplicates("preprocessing")
        .iterrows()
    )
}
PREPROCESSING_CONFIGS = normalize_preprocessing_configs(PREPROCESSING_CONFIGS)

display(
    pd.DataFrame([
        {
            "preprocessing": name,
            "preprocessing_steps": "+".join(steps),
        }
        for name, steps in PREPROCESSING_CONFIGS.items()
    ])
)

,preprocessing,preprocessing_steps
0,absorbance_sg_smooth,absorbance+sg_smooth
1,absorbance_sg_d2,absorbance+sg_d2
2,raw,raw
3,sg_smooth,sg_smooth
4,snv_sg_smooth,snv+sg_smooth
5,absorbance_snv_sg_d1,absorbance+snv+sg_d1
6,absorbance_snv_sg_smooth,absorbance+snv+sg_smooth


## Ablation study

In [7]:
ABLATION_FACTORS = [
    "matrix_family",
    "training_matrix_id",
    "matrix_method",
    "preprocessing",
    "selected_rule_name",
    "rule",
    "rule_variant",
    "n_components",
    "alpha",
    "object_threshold",
    "sg_window_length",
    "sg_polyorder",
    "position_dilation_radius",
    "balanced_pixel_strategy",
]

ablation_summary_df = summarize_ablation_effects(
    grid_summary_df,
    factor_cols=ABLATION_FACTORS,
    group_cols=["matrix_family"],
    metric_cols=[
        "balanced_accuracy",
        "target_sensitivity",
        "non_target_specificity",
        "fn_rate",
        "fp_rate",
        "selection_score",
    ],
)

save_parquet(ablation_summary_df, ABLATION_SUMMARY_PATH)

print("Ablation summary:", ablation_summary_df.shape)
print("Saved:", ABLATION_SUMMARY_PATH)

display(ablation_summary_df.head(80))

Ablation summary: (120, 48)
Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_validation_robustness_non_noisy_all\ablation_summary.parquet


,matrix_family,factor,factor_value,factor_value_numeric,n_configs,balanced_accuracy_mean,balanced_accuracy_median,balanced_accuracy_std,balanced_accuracy_min,balanced_accuracy_max,target_sensitivity_mean,target_sensitivity_median,target_sensitivity_std,target_sensitivity_min,target_sensitivity_max,non_target_specificity_mean,non_target_specificity_median,non_target_specificity_std,non_target_specificity_min,non_target_specificity_max,fn_rate_mean,fn_rate_median,fn_rate_std,fn_rate_min,fn_rate_max,fp_rate_mean,fp_rate_median,fp_rate_std,fp_rate_min,fp_rate_max,selection_score_mean,selection_score_median,selection_score_std,selection_score_min,selection_score_max,training_matrix_id,matrix_method,preprocessing,selected_rule_name,rule,rule_variant,n_components,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,balanced_pixel_strategy
0,object_matrix,alpha,0.009999999776482582,0.01,10530,0.516738,0.500000,0.037803,0.439108,0.757118,0.154377,0.000000,0.271478,0.000000,1.000000,0.879099,1.000000,0.242052,0.000000,1.000000,0.845623,1.000000,0.271478,0.000000,1.000000,0.120901,0.000000,0.242052,0.000000,1.000000,-8.559106,-9.989815,2.494408,-10.044916,-0.901540,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.009999999776482582,<NA>,<NA>,<NA>,<NA>,<NA>
1,object_matrix,alpha,0.05000000074505806,0.05,10530,0.511999,0.500000,0.036647,0.442539,0.774614,0.034459,0.000000,0.104988,0.000000,0.754717,0.989540,1.000000,0.046963,0.363636,1.000000,0.965541,1.000000,0.104988,0.245283,1.000000,0.010460,0.000000,0.046963,0.000000,0.636364,-9.653034,-9.989815,1.020085,-10.063283,-2.956621,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0.05000000074505806,<NA>,<NA>,<NA>,<NA>,<NA>
2,object_matrix,balanced_pixel_strategy,not_applicable,NaN,21060,0.514368,0.500000,0.037305,0.439108,0.774614,0.094418,0.000000,0.214374,0.000000,1.000000,0.934319,1.000000,0.182884,0.000000,1.000000,0.905582,1.000000,0.214374,0.000000,1.000000,0.065681,0.000000,0.182884,0.000000,1.000000,-9.106070,-9.989815,1.982546,-10.063283,-0.901540,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,not_applicable
3,object_matrix,matrix_family,object_matrix,NaN,21060,0.514368,0.500000,0.037305,0.439108,0.774614,0.094418,0.000000,0.214374,0.000000,1.000000,0.934319,1.000000,0.182884,0.000000,1.000000,0.905582,1.000000,0.214374,0.000000,1.000000,0.065681,0.000000,0.182884,0.000000,1.000000,-9.106070,-9.989815,1.982546,-10.063283,-0.901540,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,object_matrix,matrix_method,object_median,NaN,10530,0.526952,0.500000,0.048008,0.439108,0.774614,0.179722,0.018868,0.271224,0.000000,1.000000,0.874183,1.000000,0.239678,0.000000,1.000000,0.820278,0.981132,0.271224,0.000000,1.000000,0.125817,0.000000,0.239678,0.000000,1.000000,-8.308519,-9.835899,2.501339,-10.063283,-0.920118,<NA>,object_median,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,object_matrix,matrix_method,object_mean,NaN,10530,0.501784,0.500000,0.012721,0.446484,0.740995,0.009113,0.000000,0.061616,0.000000,1.000000,0.994456,1.000000,0.047061,0.018182,1.000000,0.990887,1.000000,0.061616,0.000000,1.000000,0.005544,0.000000,0.047061,0.000000,0.981818,-9.903621,-9.989815,0.576291,-10.026549,-0.901540,<NA>,object_mean,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,object_matrix,n_components,3,3.00,2340,0.522186,0.500000,0.045151,0.439108,0.746312,0.117038,0.000000,0.221448,0.000000,0.981132,0.927335,1.000000,0.174807,0.090909,1.000000,0.882962,1.000000,0.221448,0.018868,1.000000,0.072665,0.000000,0.174807,0.000000,0.909091,-8.885151,-9.989815,2.065353,-10.026549,-0.923510,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,object_matrix,n_components,4,4.00,2340,0.521309,0.500000,0.046409,0.455918,0.745969,0.097823,0.000000,0.205834,0.000000,0.886792,0.944794,1.000000,0.155655,0.145455,1.000000,0.902177,1.000000,0.205834,0.113208,1.000000,0.055206,0.000000,0.155655,0.000000,0.854545,-9.060835,-9.989815,1.929329,-10.044916,-1.944463,<NA>,<NA

In [8]:
ablation_plot_df = ablation_summary_df.copy()

ablation_plot_df["ablation_factor"] = (
    ablation_plot_df["factor"].astype(str)
    + "="
    + ablation_plot_df["factor_value"].astype(str)
)

for metric in (
    "fn_rate",
    "fp_rate",
    "balanced_accuracy",
):
    mean_col = f"{metric}_mean"

    if mean_col not in ablation_plot_df.columns:
        continue

    ablation_plot_df[metric] = pd.to_numeric(
        ablation_plot_df[mean_col],
        errors="coerce",
    )

    ablation_plot_df[f"delta_{metric}"] = (
        ablation_plot_df[metric]
        - ablation_plot_df
        .groupby("matrix_family")[metric]
        .transform("mean")
    )

plot_ablation_deltas(
    ablation_plot_df,
    factor_col="ablation_factor",
    metric_cols=(
        "fn_rate",
        "fp_rate",
        "balanced_accuracy",
    ),
    group_cols=("matrix_family",),
    title="Parameter-level effects relative to family mean",
    show=True,
)

## Validation refit

In [9]:
(
    validation_refit_metrics_df,
    validation_refit_objects_df,
    validation_refit_pixels_df,
    validation_pixel_errors_by_image_df,
    validation_refit_errors_df,
) = refit_selected_simca_configs(
    selected_configs_df=candidate_panel_df,
    object_db=object_db,
    image_db=image_db,
    train_filters=TRAIN_FILTERS,
    projection_filters=VALIDATION_FILTERS,
    preprocessing_configs=PREPROCESSING_CONFIGS,
    evaluation_split="validation_batch_3_refit",
    wavelengths=wavelengths,
    random_state=RANDOM_STATE,
    replace=REPLACE_BALANCED_PIXELS,
    cv_n_splits=CV_N_SPLITS,
    cv_group_col=CV_GROUP_COL,
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
)

validation_refit_objects_df = add_binary_confusion_case(
    validation_refit_objects_df,
    target_class=TARGET_CLASS,
    level="object",
)

validation_refit_pixels_df = add_binary_confusion_case(
    validation_refit_pixels_df,
    target_class=TARGET_CLASS,
    level="pixel",
)

save_parquet(validation_refit_metrics_df, VALIDATION_REFIT_METRICS_PATH)
save_parquet(validation_refit_objects_df, VALIDATION_REFIT_OBJECTS_PATH)
save_parquet(validation_refit_pixels_df, VALIDATION_REFIT_PIXELS_PATH)
save_parquet(validation_pixel_errors_by_image_df, VALIDATION_PIXEL_ERRORS_BY_IMAGE_PATH)
save_parquet_if_nonempty(validation_refit_errors_df, VALIDATION_REFIT_ERRORS_PATH)

print("Validation refit metrics:", validation_refit_metrics_df.shape)
print("Validation objects:", validation_refit_objects_df.shape)
print("Validation pixels:", validation_refit_pixels_df.shape)
print("Validation errors:", validation_refit_errors_df.shape)

display(validation_refit_metrics_df.head())
display(validation_refit_objects_df.head())
display(validation_refit_pixels_df.head())
display(validation_pixel_errors_by_image_df.head())
display(validation_refit_errors_df)

[validation_batch_3_refit] 04A_object_matrix_0001
[validation_batch_3_refit] 04A_object_matrix_0002
[validation_batch_3_refit] 04A_object_matrix_0003
[validation_batch_3_refit] 04A_object_matrix_0004
[validation_batch_3_refit] 04A_object_matrix_0005
[validation_batch_3_refit] 04A_object_matrix_0006
[validation_batch_3_refit] 04A_object_matrix_0007
[validation_batch_3_refit] 04A_object_matrix_0008
[validation_batch_3_refit] 04A_object_matrix_0009
[validation_batch_3_refit] 04A_object_matrix_0010
[validation_batch_3_refit] 04A_object_matrix_0011
[validation_batch_3_refit] 04A_object_matrix_0012
[validation_batch_3_refit] 04A_object_matrix_0013
[validation_batch_3_refit] 04A_object_matrix_0014
[validation_batch_3_refit] 04A_object_matrix_0015
[validation_batch_3_refit] 04A_object_matrix_0016
[validation_batch_3_refit] 04A_pixel_matrix_0001
[validation_batch_3_refit] 04A_pixel_matrix_0002
[validation_batch_3_refit] 04A_pixel_matrix_0003
[validation_batch_3_refit] 04A_pixel_matrix_0004
[val

,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,object_threshold,selection_score,search_method,model_family,matrix_family,training_matrix_id,matrix_method,m,m_effective,balanced_pixel_strategy,balanced_pixel_strategy_effective,preprocessing,preprocessing_steps,rule_variant,rule,n_components,alpha,non_target_label,sg_window_length,sg_polyorder,position_dilation_radius,cv_n_splits,n_cv_observations,n_cv_groups,H_emp_cv,Q_emp_cv,simple_emp_cv,alternative_chi2_emp_cv,alternative_empHQ_emp_cv,data_driven_emp_cv,cv_target_rejection_rate,cv_target_acceptance_rate,cv_expected_rejection_rate,cv_abs_rejection_error,cv_rule_limit,rule_original,rule_variant_original,rule_token,selected_rule_name,rule_for_refit,limit_source,selection_split,selection_strategy,grid_config_id,kept_after_within_same_model_preprocessing,kept_after_within_same_family_components,kept_after_family_level_pareto,selected_config_id,three_way_lower_threshold,three_way_upper_threshold,val3_target_miss_rate,val3_false_accept_rate,val3_uncertain_rate,val3_coverage_rate,score_conservative_target,score_balanced_reference,score_specificity_control,evaluation_split,n_projected_objects,n_projected_pixels
0,peanut,almond,108,53,0,52,3,1.000000,0.054545,0.527273,0.518519,0.504762,0.670886,0.000000,0.945455,0.75,-0.901540,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_mean,object_mean,40,40,not_applicable,random,absorbance_sg_smooth,absorbance+sg_smooth,data_driven_emp_cv,data_driven,11,0.01,almond,11,2,3,5,98,98,66.668457,5.685526e-05,16.527746,17.928278,1.512749,1333.969360,0.010204,0.989796,0.01,0.000204,1333.969360,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,grid_001689,True,True,True,04A_object_matrix_0001,0.05,0.95,0.0,0.545455,0.435185,0.564815,-1.363636,1.566509,-2.672727,validation_batch_3_refit,108,6812
1,peanut,almond,108,52,1,43,12,0.981132,0.218182,0.599657,0.592593,0.547368,0.702703,0.018868,0.781818,0.70,-0.923510,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,40,40,not_applicable,random,absorbance_sg_d2,absorbance+sg_d2,data_driven_emp_cv,data_driven,3,0.01,almond,11,2,3,5,98,98,17.640045,2.147866e-09,4.319395,4.536808,1.380628,384.212128,0.010204,0.989796,0.01,0.000204,384.212128,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,grid_009680,True,True,True,04A_object_matrix_0002,0.55,0.95,0.0,0.218182,0.712963,0.287037,-1.341338,1.959548,-1.785249,validation_batch_3_refit,108,6812
2,peanut,almond,108,50,3,41,14,0.943396,0.254545,0.598971,0.592593,0.549451,0.694444,0.056604,0.745455,0.75,-1.264918,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,40,40,not_applicable,random,absorbance_sg_d2,absorbance+sg_d2,data_driven_emp_cv,data_driven,3,0.01,almond,11,2,3,5,98,98,17.640045,2.147866e-09,4.319395,4.536808,1.380628,384.212128,0.010204,0.989796,0.01,0.000204,384.212128,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,grid_011947,True,True,True,04A_object_matrix_0003,0.55,0.95,0.0,0.218182,0.712963,0.287037,-2.024014,1.872387,-1.755746,validation_batch_3_refit,108,6812
3,peanut,almond,108,49,4,37,18,0.924528,0.327273,0.625901,0.620370,0.569767,0.705036,0.075472,0.672727,0.70,-1.379785,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_median,object_median,40,40,not_applicable,random,absorbance_sg_d2,absorbance+sg_d2,simple_emp_cv,simple,3,0.01,almond,11,2,3,5,98,98,17.640045,2.147866e-09,4.319395,4.536808,1.380628,384.212128,0.010204,0.989796,0.01,0.000204,4.319395,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,validation_batch_3,04A_gri

,object_id,source_image,n_pixels_projected,n_predicted_peanut_pixels,peanut_pixel_ratio,H_mean,Q_mean,H_norm_limit_mean,Q_norm_limit_mean,rule_statistic_mean,rule_limit_mean,true_peanut_pixel_ratio,truth_available_ratio,predicted_peanut_object,predicted_label_object,object_threshold,true_peanut_object,true_label_object,area_pixels,batch,sample_kind,object_nut_type,centroid_row,centroid_col,selected_config_id,matrix_family,training_matrix_id,matrix_method,balanced_pixel_strategy,balanced_pixel_strategy_effective,model_family,preprocessing,preprocessing_steps,rule,rule_variant,selected_rule_name,rule_for_refit,target_class,non_target_label,n_components,alpha,m,m_effective,sg_window_length,sg_polyorder,position_dilation_radius,evaluation_split,object_error_case,object_is_error,object_is_fp,object_is_fn
0,almond3_obj001,almond3,63,61,0.968254,252.441662,0.000158,10.315246,18.733340,559.052804,1333.969396,0.0,1.0,True,peanut,0.75,False,almond,63,3,pure,almond,77.349206,85.730159,04A_object_matrix_0001,object_matrix,object_mean,object_mean,not_applicable,random,empirical_cv_rule,absorbance_sg_smooth,absorbance+sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,peanut,almond,11,0.01,40,40,11,2,3,validation_batch_3_refit,FP,True,True,False
1,almond3_obj002,almond3,108,105,0.972222,150.927921,0.000120,6.167202,14.260540,383.906963,1333.969396,0.0,1.0,True,peanut,0.75,False,almond,108,3,pure,almond,84.518519,136.824074,04A_object_matrix_0001,object_matrix,object_mean,object_mean,not_applicable,random,empirical_cv_rule,absorbance_sg_smooth,absorbance+sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,peanut,almond,11,0.01,40,40,11,2,3,validation_batch_3_refit,FP,True,True,False
2,almond3_obj003,almond3,83,78,0.939759,276.277967,0.000134,11.289243,15.894325,537.062690,1333.969396,0.0,1.0,True,peanut,0.75,False,almond,83,3,pure,almond,84.638554,182.626506,04A_object_matrix_0001,object_matrix,object_mean,object_mean,not_applicable,random,empirical_cv_rule,absorbance_sg_smooth,absorbance+sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,peanut,almond,11,0.01,40,40,11,2,3,validation_batch_3_refit,FP,True,True,False
3,almond3_obj004,almond3,90,87,0.966667,209.520663,0.000112,8.561412,13.276159,427.129023,1333.969396,0.0,1.0,True,peanut,0.75,False,almond,90,3,pure,almond,87.466667,50.633333,04A_object_matrix_0001,object_matrix,object_mean,object_mean,not_applicable,random,empirical_cv_rule,absorbance_sg_smooth,absorbance+sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,peanut,almond,11,0.01,40,40,11,2,3,validation_batch_3_refit,FP,True,True,False
4,almond3_obj005,almond3,58,44,0.758621,844.826142,0.000251,34.521202,29.768371,1336.623169,1333.969396,0.0,1.0,True,peanut,0.75,False,almond,58,3,pure,almond,86.896552,231.965517,04A_object_matrix_0001,object_matrix,object_mean,object_mean,not_applicable,random,empirical_cv_rule,absorbance_sg_smooth,absorbance+sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,peanut,almond,11,0.01,40,40,11,2,3,validation_batch_3_refit,FP,True,True,False


,object_id,label,source_image,batch,area,sample_kind,pixel_index,row,col,H,Q,H_norm_limit,Q_norm_limit,matrix_method,target_class,T1,T2,T3,T4,T5,T6,T7,T8,T9,T10,T11,stat_data_driven_emp_cv,limit_data_driven_emp_cv,pred_data_driven_emp_cv,true_peanut_pixel,truth_available,predicted_peanut_pixel,predicted_label_pixel,rule_statistic,rule_limit,rule_name,non_target_label,selected_config_id,matrix_family,training_matrix_id,balanced_pixel_strategy,balanced_pixel_strategy_effective,model_family,preprocessing,preprocessing_steps,rule,rule_variant,selected_rule_name,rule_for_refit,n_components,alpha,object_threshold,m,m_effective,sg_window_length,sg_polyorder,position_dilation_radius,evaluation_split,stat_simple_emp_cv,limit_simple_emp_cv,pred_simple_emp_cv,stat_alternative_chi2_emp_cv,limit_alternative_chi2_emp_cv,pred_alternative_chi2_emp_cv,stat_alternative_chi2_fixed2,limit_alternative_chi2_fixed2,pred_alternative_chi2_fixed2,stat_data_driven_chi2,limit_data_driven_chi2,pred_data_driven_chi2,stat_simple_chi2,limit_simple_chi2,pred_simple_chi2,pixel_error_case,pixel_is_error,pixel_is_fp,pixel_is_fn
0,almond3_obj001,almond,almond3,3,63,pure,0,73,87,1296.917674,0.000335,52.994521,39.659949,object_mean,peanut,2.218170,0.087554,0.274052,0.226050,0.079445,-0.091734,0.086405,-0.031536,0.050840,0.008362,-0.014851,1953.898083,1333.969396,False,False,True,False,almond,1953.898083,1333.969396,data_driven_emp_cv,almond,04A_object_matrix_0001,object_matrix,object_mean,not_applicable,random,empirical_cv_rule,absorbance_sg_smooth,absorbance+sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,11,0.01,0.75,40,40,11,2,3,validation_batch_3_refit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TN,False,False,False
1,almond3_obj001,almond,almond3,3,63,pure,1,73,88,411.485529,0.000194,16.814081,22.967515,object_mean,peanut,1.793233,0.016585,0.126418,0.081077,0.049853,-0.037251,0.064457,-0.017211,0.027830,0.001536,-0.008756,788.449390,1333.969396,True,False,True,True,peanut,788.449390,1333.969396,data_driven_emp_cv,almond,04A_object_matrix_0001,object_matrix,object_mean,not_applicable,random,empirical_cv_rule,absorbance_sg_smooth,absorbance+sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,11,0.01,0.75,40,40,11,2,3,validation_batch_3_refit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FP,True,True,False
2,almond3_obj001,almond,almond3,3,63,pure,2,73,89,244.010550,0.000152,9.970735,17.983690,object_mean,peanut,1.453705,-0.097571,0.119594,0.090234,0.043859,-0.031539,0.034179,-0.006006,0.020872,0.001700,-0.008798,538.369281,1333.969396,True,False,True,True,peanut,538.369281,1333.969396,data_driven_emp_cv,almond,04A_object_matrix_0001,object_matrix,object_mean,not_applicable,random,empirical_cv_rule,absorbance_sg_smooth,absorbance+sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,11,0.01,0.75,40,40,11,2,3,validation_batch_3_refit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FP,True,True,False
3,almond3_obj001,almond,almond3,3,63,pure,3,74,86,498.831438,0.000425,20.383201,50.348265,object_mean,peanut,1.930178,0.099660,0.107572,0.058092,0.051710,-0.048552,0.070368,-0.010891,0.034697,-0.009656,-0.007054,1321.036441,1333.969396,True,False,True,True,peanut,1321.036441,1333.969396,data_driven_emp_cv,almond,04A_object_matrix_0001,object_matrix,object_mean,not_applicable,random,empirical_cv_rule,absorbance_sg_smooth,absorbance+sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,11,0.01,0.75,40,40,11,2,3,validation_batch_3_refit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FP,True,True,False
4,almond3_obj001,almond,almond3,3,63,pure,4,74,87,153.281799,0.000200,6.263386,23.686328,object_mean,peanut,1.710593,0.191566,-0.000959,-0.036007,0.027080,-0.005839,0.035900,0.029590,0.000812,-0.003109,-0.006555,539.248822,1333.969396,True,False,True,True,peanut,539.248822,1333.969396,data_driven_emp_cv,almond,04A_

,source_image,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,n_truth_pixels,pixel_accuracy,pixel_balanced_accuracy,pixel_fn_rate,pixel_fp_rate,selected_config_id,evaluation_split
0,peanut3,peanut,almond,3197,3006,191,0,0,0.940256,NaN,NaN,0.940256,1.0,0.969208,0.059744,NaN,3197,0.940256,NaN,0.059744,NaN,04A_object_matrix_0001,validation_batch_3_refit
1,almond3,peanut,almond,3615,0,0,3357,258,NaN,0.071369,NaN,0.071369,0.0,NaN,NaN,0.928631,3615,0.071369,NaN,NaN,0.928631,04A_object_matrix_0001,validation_batch_3_refit
2,peanut3,peanut,almond,3197,2822,375,0,0,0.882703,NaN,NaN,0.882703,1.0,0.937697,0.117297,NaN,3197,0.882703,NaN,0.117297,NaN,04A_object_matrix_0002,validation_batch_3_refit
3,almond3,peanut,almond,3615,0,0,2883,732,NaN,0.202490,NaN,0.202490,0.0,NaN,NaN,0.797510,3615,0.202490,NaN,NaN,0.797510,04A_object_matrix_0002,validation_batch_3_refit
4,peanut3,peanut,almond,3197,2822,375,0,0,0.882703,NaN,NaN,0.882703,1.0,0.937697,0.117297,NaN,3197,0.882703,NaN,0.117297,NaN,04A_object_matrix_0003,validation_batch_3_refit


""


### Apply fixed 3-way thresholds selected in 04A

In [10]:
validation_3way_metrics_df, validation_3way_objects_df = evaluate_three_way_by_config(
    object_df=validation_refit_objects_df,
    thresholds_df=candidate_panel_df,
    config_id_col="selected_config_id",
    target_class=TARGET_CLASS,
    non_target_label=NON_TARGET_LABEL,
)

save_parquet(validation_3way_objects_df, VALIDATION_3WAY_OBJECTS_PATH)
save_parquet(validation_3way_metrics_df, VALIDATION_3WAY_METRICS_PATH)

display(
    validation_3way_metrics_df.sort_values(
        [
            "target_miss_rate",
            "non_target_false_accept_rate",
            "uncertain_rate",
            "coverage_rate",
        ],
        ascending=[True, True, True, False],
    ).head(30)
)

,n,n_target,n_non_target,n_uncertain,uncertain_rate,coverage_rate,target_miss_rate,screening_sensitivity,target_auto_accept_rate,target_uncertain_rate,non_target_false_accept_rate,non_target_auto_reject_rate,non_target_uncertain_rate,decided_tp,decided_fn,decided_fp,decided_tn,decided_accuracy,decided_balanced_accuracy,three_way_score,selected_config_id
21,108,53,55,16,0.148148,0.851852,0.0,1.0,0.867925,0.132075,0.000000,0.836364,0.163636,46,0,0,46,1.000000,1.000000,1.135017,04A_pixel_matrix_0006
20,108,53,55,17,0.157407,0.842593,0.0,1.0,0.811321,0.188679,0.000000,0.872727,0.127273,43,0,0,48,1.000000,1.000000,1.139478,04A_pixel_matrix_0005
18,108,53,55,18,0.166667,0.833333,0.0,1.0,0.849057,0.150943,0.000000,0.818182,0.181818,45,0,0,45,1.000000,1.000000,1.121212,04A_pixel_matrix_0003
17,108,53,55,31,0.287037,0.712963,0.0,1.0,0.584906,0.415094,0.000000,0.836364,0.163636,31,0,0,46,1.000000,1.000000,1.065572,04A_pixel_matrix_0002
16,108,53,55,34,0.314815,0.685185,0.0,1.0,0.528302,0.471698,0.000000,0.836364,0.163636,28,0,0,46,1.000000,1.000000,1.051684,04A_pixel_matrix_0001
19,108,53,55,34,0.314815,0.685185,0.0,1.0,0.528302,0.471698,0.000000,0.836364,0.163636,28,0,0,46,1.000000,1.000000,1.051684,04A_pixel_matrix_0004
14,108,53,55,61,0.564815,0.435185,0.0,1.0,0.566038,0.433962,0.036364,0.272727,0.690909,30,0,2,15,0.957447,0.941176,0.676684,04A_object_matrix_0015
15,108,53,55,62,0.574074,0.425926,0.0,1.0,0.547170,0.452830,0.072727,0.236364,0.690909,29,0,4,13,0.913043,0.882353,0.553872,04A_object_matrix_0016
12,108,53,55,61,0.564815,0.435185,0.0,1.0,0.660377,0.339623,0.090909,0.127273,0.781818,35,0,5,7,0.893617,0.791667,0.476684,04A_object_matrix_0013
13,108,53,55,62,0.574074,0.425926,0.0,1.0,0.603774,0.396226,0.090909,0.163636,0.745455,32,0,5,9,0.891304,0.821429,0.481145,04A_object_matrix_0014


In [11]:
# validation_3way_objects_df = apply_three_way_thresholds_by_config(
#     object_df=validation_refit_objects_df,
#     thresholds_df=candidate_panel_df,
#     config_id_col="selected_config_id",
#     target_class=TARGET_CLASS,
#     non_target_label=NON_TARGET_LABEL,
# )

# three_way_metric_rows = []

# for config_id, group in validation_3way_objects_df.groupby("selected_config_id", dropna=False):
#     metrics = evaluate_three_way_object_decision(
#         group,
#         target_class=TARGET_CLASS,
#         non_target_label=NON_TARGET_LABEL,
#     )
#     metrics["selected_config_id"] = config_id
#     three_way_metric_rows.append(metrics)

# validation_3way_metrics_df = pd.DataFrame(three_way_metric_rows)

# save_parquet(validation_3way_objects_df, VALIDATION_3WAY_OBJECTS_PATH)
# save_parquet(validation_3way_metrics_df, VALIDATION_3WAY_METRICS_PATH)

# display(
#     validation_3way_metrics_df.sort_values(
#         [
#             "target_miss_rate",
#             "non_target_false_accept_rate",
#             "uncertain_rate",
#             "coverage_rate",
#         ],
#         ascending=[True, True, True, False],
#     ).head(30)
# )

## Stability

### 2-way

In [12]:
# ---------------------------------------------------------------------
# Random-state stability with object tables
# ---------------------------------------------------------------------

if RUN_RANDOM_STATE_STABILITY:
    if RUN_STABILITY_ONLY_FOR_BALANCED_PIXELS:
        stability_configs_df = candidate_panel_df[
            candidate_panel_df["matrix_method"].astype(str).eq("balanced_pixels")
        ].copy()
    else:
        stability_configs_df = candidate_panel_df.copy()

    (
        random_state_stability_metrics_df,
        random_state_stability_objects_df,
        random_state_stability_pixel_errors_df,
        random_state_stability_errors_df,
    ) = run_selected_simca_random_state_stability_full(
        selected_configs_df=stability_configs_df,
        object_db=object_db,
        image_db=image_db,
        train_filters=TRAIN_FILTERS,
        projection_filters=VALIDATION_FILTERS,
        preprocessing_configs=PREPROCESSING_CONFIGS,
        seeds=RANDOM_STABILITY_SEEDS,
        evaluation_split="validation_batch_3_random_state",
        wavelengths=wavelengths,
        replace=REPLACE_BALANCED_PIXELS,
        cv_n_splits=CV_N_SPLITS,
        cv_group_col=CV_GROUP_COL,
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )
else:
    random_state_stability_metrics_df = pd.DataFrame()
    random_state_stability_objects_df = pd.DataFrame()
    random_state_stability_pixel_errors_df = pd.DataFrame()
    random_state_stability_errors_df = pd.DataFrame()

stability_summary_df = summarize_metric_stability(
    random_state_stability_metrics_df,
    config_cols=[
        "selected_config_id",
        "matrix_family",
        "training_matrix_id",
        "matrix_method",
        "preprocessing",
        "selected_rule_name",
        "rule_for_refit",
        "n_components",
        "alpha",
        "object_threshold",
    ],
    metric_cols=[
        "balanced_accuracy",
        "target_sensitivity",
        "non_target_specificity",
        "fn_rate",
        "fp_rate",
    ],
    seed_col="random_state",
)

save_parquet(stability_summary_df, RANDOM_STATE_STABILITY_PATH)
save_parquet_if_nonempty(random_state_stability_objects_df, RANDOM_STATE_STABILITY_OBJECTS_PATH)
save_parquet_if_nonempty(random_state_stability_errors_df, RANDOM_STATE_STABILITY_ERRORS_PATH)

display(stability_summary_df.head(30))

[random_state_stability_full] seed=0
[validation_batch_3_random_state] 04A_pixel_matrix_0001
[validation_batch_3_random_state] 04A_pixel_matrix_0002
[validation_batch_3_random_state] 04A_pixel_matrix_0003
[validation_batch_3_random_state] 04A_pixel_matrix_0004
[validation_batch_3_random_state] 04A_pixel_matrix_0005
[validation_batch_3_random_state] 04A_pixel_matrix_0006
[random_state_stability_full] seed=1
[validation_batch_3_random_state] 04A_pixel_matrix_0001
[validation_batch_3_random_state] 04A_pixel_matrix_0002
[validation_batch_3_random_state] 04A_pixel_matrix_0003
[validation_batch_3_random_state] 04A_pixel_matrix_0004
[validation_batch_3_random_state] 04A_pixel_matrix_0005
[validation_batch_3_random_state] 04A_pixel_matrix_0006
[random_state_stability_full] seed=2
[validation_batch_3_random_state] 04A_pixel_matrix_0001
[validation_batch_3_random_state] 04A_pixel_matrix_0002
[validation_batch_3_random_state] 04A_pixel_matrix_0003
[validation_batch_3_random_state] 04A_pixel_matri

,selected_config_id,matrix_family,training_matrix_id,matrix_method,preprocessing,selected_rule_name,rule_for_refit,n_components,alpha,object_threshold,n_runs,n_random_states,mean_balanced_accuracy,std_balanced_accuracy,min_balanced_accuracy,max_balanced_accuracy,mean_target_sensitivity,std_target_sensitivity,min_target_sensitivity,max_target_sensitivity,mean_non_target_specificity,std_non_target_specificity,min_non_target_specificity,max_non_target_specificity,mean_fn_rate,std_fn_rate,min_fn_rate,max_fn_rate,mean_fp_rate,std_fp_rate,min_fp_rate,max_fp_rate,stability_score
0,04A_pixel_matrix_0001,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,absorbance_sg_smooth,simple_chi2,simple_chi2,7,0.01,0.75,10,10,0.939760,0.010274,0.917496,0.954545,0.986792,0.014736,0.962264,1.000000,0.892727,1.715269e-02,0.854545,0.909091,0.013208,0.014736,0.000000,0.037736,0.107273,0.017153,0.090909,0.145455,0.502305
1,04A_pixel_matrix_0002,pixel_matrix,balanced_pixel_center_m40,balanced_pixels,absorbance_sg_smooth,simple_emp_cv,simple_emp_cv,7,0.05,0.70,10,10,0.935334,0.000000,0.935334,0.935334,0.943396,0.000000,0.943396,0.943396,0.927273,1.110223e-16,0.927273,0.927273,0.056604,0.000000,0.056604,0.056604,0.072727,0.000000,0.072727,0.072727,0.223842
2,04A_pixel_matrix_0003,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,absorbance_snv_sg_d1,data_driven_chi2,data_driven_chi2,11,0.01,0.70,10,10,0.920755,0.019099,0.889880,0.944082,0.941509,0.013208,0.924528,0.962264,0.900000,4.321769e-02,0.818182,0.963636,0.058491,0.013208,0.037736,0.075472,0.100000,0.043218,0.036364,0.181818,0.026594
3,04A_pixel_matrix_0004,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,absorbance_sg_smooth,simple_chi2,simple_chi2,7,0.01,0.80,10,10,0.922967,0.009491,0.907376,0.944768,0.913208,0.026949,0.886792,0.962264,0.932727,1.827250e-02,0.890909,0.945455,0.086792,0.026949,0.037736,0.113208,0.067273,0.018273,0.054545,0.109091,-0.232519
4,04A_pixel_matrix_0005,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,absorbance_snv_sg_smooth,simple_chi2,simple_chi2,6,0.05,0.75,10,10,0.921475,0.009266,0.897942,0.934305,0.881132,0.008646,0.867925,0.886792,0.961818,2.065058e-02,0.909091,0.981818,0.118868,0.008646,0.113208,0.132075,0.038182,0.020651,0.018182,0.090909,-0.407450
5,04A_pixel_matrix_0006,pixel_matrix,balanced_pixel_random_m40,balanced_pixels,absorbance_snv_sg_d1,data_driven_chi2,data_driven_chi2,11,0.05,0.70,10,10,0.919451,0.017410,0.888508,0.943396,0.869811,0.010161,0.849057,0.886792,0.969091,3.257541e-02,0.909091,1.000000,0.130189,0.010161,0.113208,0.150943,0.030909,0.032575,0.000000,0.090909,-0.527633


### 3-way

In [13]:
# ---------------------------------------------------------------------
# 3-way random-state stability using fixed 04A thresholds
# ---------------------------------------------------------------------

if (
    RUN_RANDOM_STATE_STABILITY
    and isinstance(random_state_stability_objects_df, pd.DataFrame)
    and len(random_state_stability_objects_df) > 0
    and "selected_config_id" in random_state_stability_objects_df.columns
):
    random_state_stability_3way_metrics_df, random_state_stability_3way_objects_df = evaluate_three_way_by_config(
        object_df=random_state_stability_objects_df,
        thresholds_df=candidate_panel_df,
        config_id_col="selected_config_id",
        extra_group_cols=["random_state"],
        target_class=TARGET_CLASS,
        non_target_label=NON_TARGET_LABEL,
    )

    three_way_stability_summary_df = summarize_metric_stability(
        random_state_stability_3way_metrics_df,
        config_cols=["selected_config_id"],
        metric_cols=[
            "target_miss_rate",
            "screening_sensitivity",
            "non_target_false_accept_rate",
            "uncertain_rate",
            "coverage_rate",
            "non_target_auto_reject_rate",
            "decided_balanced_accuracy",
        ],
        seed_col="random_state",
    )

else:
    random_state_stability_3way_metrics_df = pd.DataFrame()
    random_state_stability_3way_objects_df = pd.DataFrame()
    three_way_stability_summary_df = pd.DataFrame()
    print("No 3-way random-state stability table produced.")

save_parquet_if_nonempty(
    random_state_stability_3way_metrics_df,
    RANDOM_STATE_STABILITY_3WAY_METRICS_PATH,
)

save_parquet_if_nonempty(
    three_way_stability_summary_df,
    RANDOM_STATE_STABILITY_3WAY_SUMMARY_PATH,
)

display(three_way_stability_summary_df.head(30))

,selected_config_id,n_runs,n_random_states,mean_target_miss_rate,std_target_miss_rate,min_target_miss_rate,max_target_miss_rate,mean_screening_sensitivity,std_screening_sensitivity,min_screening_sensitivity,max_screening_sensitivity,mean_non_target_false_accept_rate,std_non_target_false_accept_rate,min_non_target_false_accept_rate,max_non_target_false_accept_rate,mean_uncertain_rate,std_uncertain_rate,min_uncertain_rate,max_uncertain_rate,mean_coverage_rate,std_coverage_rate,min_coverage_rate,max_coverage_rate,mean_non_target_auto_reject_rate,std_non_target_auto_reject_rate,min_non_target_auto_reject_rate,max_non_target_auto_reject_rate,mean_decided_balanced_accuracy,std_decided_balanced_accuracy,min_decided_balanced_accuracy,max_decided_balanced_accuracy
0,04A_pixel_matrix_0001,10,10,0.000000,0.000000,0.0,0.000000,1.000000,0.000000,1.000000,1.0,0.001818,0.005455,0.0,0.018182,0.327778,1.504452e-02,0.314815,0.370370,0.672222,0.015045,0.629630,0.685185,0.818182,0.021513,0.781818,0.836364,0.998864,0.003409,0.988636,1.0
1,04A_pixel_matrix_0002,10,10,0.000000,0.000000,0.0,0.000000,1.000000,0.000000,1.000000,1.0,0.000000,0.000000,0.0,0.000000,0.287037,5.551115e-17,0.287037,0.287037,0.712963,0.000000,0.712963,0.712963,0.836364,0.000000,0.836364,0.836364,1.000000,0.000000,1.000000,1.0
2,04A_pixel_matrix_0003,10,10,0.020755,0.015673,0.0,0.037736,0.979245,0.015673,0.962264,1.0,0.040000,0.046568,0.0,0.109091,0.131481,2.335096e-02,0.092593,0.166667,0.868519,0.023351,0.833333,0.907407,0.776364,0.067567,0.690909,0.854545,0.963745,0.025079,0.931444,1.0
3,04A_pixel_matrix_0004,10,10,0.000000,0.000000,0.0,0.000000,1.000000,0.000000,1.000000,1.0,0.001818,0.005455,0.0,0.018182,0.327778,1.504452e-02,0.314815,0.370370,0.672222,0.015045,0.629630,0.685185,0.818182,0.021513,0.781818,0.836364,0.998864,0.003409,0.988636,1.0
4,04A_pixel_matrix_0005,10,10,0.005660,0.008646,0.0,0.018868,0.994340,0.008646,0.981132,1.0,0.016364,0.020651,0.0,0.054545,0.183333,3.813196e-02,0.148148,0.287037,0.816667,0.038132,0.712963,0.851852,0.778182,0.096484,0.527273,0.872727,0.985058,0.013900,0.953125,1.0
5,04A_pixel_matrix_0006,10,10,0.001887,0.005660,0.0,0.018868,0.998113,0.005660,0.981132,1.0,0.030909,0.032575,0.0,0.090909,0.175000,2.822176e-02,0.138889,0.212963,0.825000,0.028222,0.787037,0.861111,0.749091,0.083240,0.618182,0.836364,0.977725,0.022612,0.937500,1.0


In [14]:
plot_stability_intervals(
    random_state_stability_3way_metrics_df,
    config_col="selected_config_id",
    metric_col="target_miss_rate",
    seed_col="random_state",
    family_col="matrix_family",
    title="Random-state stability — 3-way target miss rate",
    show=True,
)

In [15]:
# # ---------------------------------------------------------------------
# # 3-way random-state stability using fixed 04A thresholds
# # ---------------------------------------------------------------------

# if len(random_state_stability_objects_df) > 0:
#     random_state_stability_3way_objects_df = apply_three_way_thresholds_by_config(
#         object_df=random_state_stability_objects_df,
#         thresholds_df=candidate_panel_df,
#         config_id_col="selected_config_id",
#         target_class=TARGET_CLASS,
#         non_target_label=NON_TARGET_LABEL,
#     )

#     three_way_seed_rows = []

#     for (config_id, seed), group in random_state_stability_3way_objects_df.groupby(
#         ["selected_config_id", "random_state"],
#         dropna=False,
#     ):
#         metrics = evaluate_three_way_object_decision(
#             group,
#             target_class=TARGET_CLASS,
#             non_target_label=NON_TARGET_LABEL,
#         )
#         metrics["selected_config_id"] = config_id
#         metrics["random_state"] = int(seed)
#         three_way_seed_rows.append(metrics)

#     random_state_stability_3way_metrics_df = pd.DataFrame(three_way_seed_rows)

# else:
#     random_state_stability_3way_objects_df = pd.DataFrame()
#     random_state_stability_3way_metrics_df = pd.DataFrame()


# three_way_stability_summary_df = summarize_metric_stability(
#     random_state_stability_3way_metrics_df,
#     config_cols=["selected_config_id"],
#     metric_cols=[
#         "target_miss_rate",
#         "screening_sensitivity",
#         "non_target_false_accept_rate",
#         "uncertain_rate",
#         "coverage_rate",
#         "non_target_auto_reject_rate",
#         "decided_balanced_accuracy",
#     ],
#     seed_col="random_state",
# )

# save_parquet_if_nonempty(
#     random_state_stability_3way_metrics_df,
#     RANDOM_STATE_STABILITY_3WAY_METRICS_PATH,
# )

# save_parquet_if_nonempty(
#     three_way_stability_summary_df,
#     RANDOM_STATE_STABILITY_3WAY_SUMMARY_PATH,
# )

# display(three_way_stability_summary_df.head(30))

In [16]:
if RUN_BORDER_DIAGNOSTIC:
    border_configs_df = (
        candidate_panel_df
        .sort_values(
            ["matrix_family", "fn_rate", "fp_rate", "balanced_accuracy"],
            ascending=[True, True, True, False],
        )
        .groupby("matrix_family", group_keys=False, dropna=False)
        .head(BORDER_DIAGNOSTIC_CONFIG_LIMIT_PER_FAMILY)
        .copy()
    )

    border_configs = (
        border_configs_df["selected_config_id"]
        .astype(str)
        .tolist()
    )

    validation_pixels_for_border_df = validation_refit_pixels_df[
        validation_refit_pixels_df["selected_config_id"].astype(str).isin(border_configs)
    ].copy()

    border_diagnostic_validation_df = summarize_border_diagnostics_by_config(
        pixel_df=validation_pixels_for_border_df,
        object_db=object_db,
        target_class=TARGET_CLASS,
        border_widths=BORDER_DIAGNOSTIC_WIDTHS,
        config_cols=[
            "selected_config_id",
            "matrix_family",
            "training_matrix_id",
            "matrix_method",
            "preprocessing",
            "selected_rule_name",
            "n_components",
            "alpha",
            "object_threshold",
        ],
    )

else:
    border_diagnostic_validation_df = pd.DataFrame()

save_parquet_if_nonempty(
    border_diagnostic_validation_df,
    BORDER_DIAGNOSTIC_VALIDATION_PATH,
)
print("Border diagnostic:", border_diagnostic_validation_df.shape)

display(border_diagnostic_validation_df.head(60))

Border diagnostic: (132, 21)


,selected_config_id,matrix_family,training_matrix_id,matrix_method,preprocessing,selected_rule_name,n_components,alpha,object_threshold,border_width,zone,n_pixels,tp,tn,fp,fn,n_errors,error_rate,fp_rate,fn_rate,pixel_accuracy
0,04A_object_matrix_0001,object_matrix,object_mean,object_mean,absorbance_sg_smooth,data_driven_emp_cv,11,0.01,0.75,1,border,2026,846,191,833,156,989,0.488154,0.813477,0.155689,0.511846
1,04A_object_matrix_0001,object_matrix,object_mean,object_mean,absorbance_sg_smooth,data_driven_emp_cv,11,0.01,0.75,1,core,4786,2160,67,2524,35,2559,0.534684,0.974141,0.015945,0.465316
2,04A_object_matrix_0001,object_matrix,object_mean,object_mean,absorbance_sg_smooth,data_driven_emp_cv,11,0.01,0.75,2,border,3984,1772,231,1803,178,1981,0.497239,0.886431,0.091282,0.502761
3,04A_object_matrix_0001,object_matrix,object_mean,object_mean,absorbance_sg_smooth,data_driven_emp_cv,11,0.01,0.75,2,core,2828,1234,27,1554,13,1567,0.554102,0.982922,0.010425,0.445898
4,04A_object_matrix_0001,object_matrix,object_mean,object_mean,absorbance_sg_smooth,data_driven_emp_cv,11,0.01,0.75,3,border,5847,2632,250,2778,187,2965,0.507098,0.917437,0.066336,0.492902
5,04A_object_matrix_0001,object_matrix,object_mean,object_mean,absorbance_sg_smooth,data_driven_emp_cv,11,0.01,0.75,3,core,965,374,8,579,4,583,0.604145,0.986371,0.010582,0.395855
6,04A_object_matrix_0002,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,3,0.01,0.70,1,border,2026,794,251,773,208,981,0.484205,0.754883,0.207585,0.515795
7,04A_object_matrix_0002,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,3,0.01,0.70,1,core,4786,2028,481,2110,167,2277,0.475763,0.814357,0.076082,0.524237
8,04A_object_matrix_0002,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,3,0.01,0.70,2,border,3984,1663,429,1605,287,1892,0.474900,0.789086,0.147179,0.525100
9,04A_object_matrix_0002,object_matrix,object_median,object_median,absorbance_sg_d2,data_driven_emp_cv,3,0.01,0.70,2,core,2828,1159,303,1278,88,1366,0.483027,0.808349,0.070569,0.516973


In [17]:
plot_border_core_metrics(
    border_diagnostic_validation_df,
    border_width_col="border_width",
    zone_col="zone",
    metric_cols=(
        "fn_rate",
        "fp_rate",
        "pixel_accuracy",
    ),
    config_col="selected_config_id",
    title="Validation — border versus core diagnostics",
    show=True,
)

In [18]:
# ---------------------------------------------------------------------
# Build robustness summary table
# ---------------------------------------------------------------------

robustness_summary_df = candidate_panel_df.copy()

# 2-way validation refit metrics.
metric_cols_to_merge = [
    "selected_config_id",
    "n",
    "tp",
    "fn",
    "fp",
    "tn",
    "balanced_accuracy",
    "target_sensitivity",
    "non_target_specificity",
    "fn_rate",
    "fp_rate",
    "f1_score",
    "accuracy",
    "precision",
    "n_projected_objects",
    "n_projected_pixels",
]

metric_cols_to_merge = [
    col for col in metric_cols_to_merge
    if col in validation_refit_metrics_df.columns
]

validation_metrics_for_merge = (
    validation_refit_metrics_df[metric_cols_to_merge]
    .rename(
        columns={
            "n": "validation_n",
            "tp": "validation_tp",
            "fn": "validation_fn",
            "fp": "validation_fp",
            "tn": "validation_tn",
            "balanced_accuracy": "validation_balanced_accuracy",
            "target_sensitivity": "validation_target_sensitivity",
            "non_target_specificity": "validation_non_target_specificity",
            "fn_rate": "validation_fn_rate",
            "fp_rate": "validation_fp_rate",
            "f1_score": "validation_f1_score",
            "accuracy": "validation_accuracy",
            "precision": "validation_precision",
        }
    )
)

robustness_summary_df = robustness_summary_df.merge(
    validation_metrics_for_merge,
    on="selected_config_id",
    how="left",
)

# 3-way validation metrics.
if len(validation_3way_metrics_df) > 0:
    validation_3way_for_merge_df = (
        validation_3way_metrics_df[
            [
                "selected_config_id",
                "target_miss_rate",
                "screening_sensitivity",
                "non_target_false_accept_rate",
                "uncertain_rate",
                "coverage_rate",
                "non_target_auto_reject_rate",
                "decided_balanced_accuracy",
            ]
        ]
        .rename(
            columns={
                "target_miss_rate": "validation_3way_target_miss_rate",
                "screening_sensitivity": "validation_3way_screening_sensitivity",
                "non_target_false_accept_rate": "validation_3way_non_target_false_accept_rate",
                "uncertain_rate": "validation_3way_uncertain_rate",
                "coverage_rate": "validation_3way_coverage_rate",
                "non_target_auto_reject_rate": "validation_3way_non_target_auto_reject_rate",
                "decided_balanced_accuracy": "validation_3way_decided_balanced_accuracy",
            }
        )
    )

    robustness_summary_df = robustness_summary_df.merge(
        validation_3way_for_merge_df,
        on="selected_config_id",
        how="left",
    )

# 2-way seed stability.
if len(stability_summary_df) > 0:
    stability_cols = [
        col for col in stability_summary_df.columns
        if col == "selected_config_id"
        or col.startswith("mean_")
        or col.startswith("std_")
        or col.startswith("min_")
        or col.startswith("max_")
        or col.startswith("n_")
    ]

    robustness_summary_df = robustness_summary_df.merge(
        stability_summary_df[stability_cols],
        on="selected_config_id",
        how="left",
    )

# 3-way seed stability.
if len(three_way_stability_summary_df) > 0:
    three_way_stability_cols = [
        col for col in three_way_stability_summary_df.columns
        if col == "selected_config_id"
        or col.startswith("mean_")
        or col.startswith("std_")
        or col.startswith("min_")
        or col.startswith("max_")
        or col.startswith("n_")
    ]

    # Avoid name collisions with 2-way stability columns.
    rename_3way_stability = {
        col: f"threeway_{col}"
        for col in three_way_stability_cols
        if col != "selected_config_id"
    }

    robustness_summary_df = robustness_summary_df.merge(
        three_way_stability_summary_df[three_way_stability_cols].rename(
            columns=rename_3way_stability
        ),
        on="selected_config_id",
        how="left",
    )

# ---------------------------------------------------------------------
# Fallback stability values for configs not checked by random-state
# ---------------------------------------------------------------------

fallback_map = {
    "mean_fn_rate": "validation_fn_rate",
    "max_fn_rate": "validation_fn_rate",
    "mean_fp_rate": "validation_fp_rate",
    "max_fp_rate": "validation_fp_rate",
    "mean_balanced_accuracy": "validation_balanced_accuracy",
}

for out_col, val_col in fallback_map.items():
    if out_col not in robustness_summary_df.columns:
        robustness_summary_df[out_col] = np.nan
    robustness_summary_df[out_col] = robustness_summary_df[out_col].fillna(
        robustness_summary_df[val_col]
    )

for std_col in ["std_fn_rate", "std_fp_rate"]:
    if std_col not in robustness_summary_df.columns:
        robustness_summary_df[std_col] = 0.0
    robustness_summary_df[std_col] = robustness_summary_df[std_col].fillna(0.0)

fallback_3way_map = {
    "threeway_mean_target_miss_rate": "validation_3way_target_miss_rate",
    "threeway_max_target_miss_rate": "validation_3way_target_miss_rate",
    "threeway_mean_non_target_false_accept_rate": "validation_3way_non_target_false_accept_rate",
    "threeway_mean_uncertain_rate": "validation_3way_uncertain_rate",
    "threeway_mean_coverage_rate": "validation_3way_coverage_rate",
}

for out_col, val_col in fallback_3way_map.items():
    if out_col not in robustness_summary_df.columns:
        robustness_summary_df[out_col] = np.nan
    if val_col in robustness_summary_df.columns:
        robustness_summary_df[out_col] = robustness_summary_df[out_col].fillna(
            robustness_summary_df[val_col]
        )

if "threeway_std_uncertain_rate" not in robustness_summary_df.columns:
    robustness_summary_df["threeway_std_uncertain_rate"] = 0.0
robustness_summary_df["threeway_std_uncertain_rate"] = robustness_summary_df[
    "threeway_std_uncertain_rate"
].fillna(0.0)


# Border diagnostic merge.
if len(border_diagnostic_validation_df) > 0:
    selected_width = max(BORDER_DIAGNOSTIC_WIDTHS)

    border_focus_df = border_diagnostic_validation_df[
        border_diagnostic_validation_df["border_width"].eq(selected_width)
    ].copy()

    border_pivot_df = border_focus_df.pivot_table(
        index="selected_config_id",
        columns="zone",
        values=["fn_rate", "fp_rate", "n_errors", "n_pixels"],
        aggfunc="first",
    )

    border_pivot_df.columns = [
        f"{metric}_{zone}_bw{selected_width}"
        for metric, zone in border_pivot_df.columns
    ]

    border_pivot_df = border_pivot_df.reset_index()

    robustness_summary_df = robustness_summary_df.merge(
        border_pivot_df,
        on="selected_config_id",
        how="left",
    )

save_parquet(robustness_summary_df, ROBUSTNESS_SUMMARY_PATH)

print("Robustness summary:", robustness_summary_df.shape)
display(robustness_summary_df.head(30))

Robustness summary: (22, 155)


,target_class,non_target_class,n,tp,fn,fp,tn,target_sensitivity,non_target_specificity,balanced_accuracy,accuracy,precision,f1_score,fn_rate,fp_rate,object_threshold,selection_score,search_method,model_family,matrix_family,training_matrix_id,matrix_method,m,m_effective,balanced_pixel_strategy,balanced_pixel_strategy_effective,preprocessing,preprocessing_steps,rule_variant,rule,n_components_x,alpha,non_target_label,sg_window_length,sg_polyorder,position_dilation_radius,cv_n_splits,n_cv_observations,n_cv_groups,H_emp_cv,Q_emp_cv,simple_emp_cv,alternative_chi2_emp_cv,alternative_empHQ_emp_cv,data_driven_emp_cv,cv_target_rejection_rate,cv_target_acceptance_rate,cv_expected_rejection_rate,cv_abs_rejection_error,cv_rule_limit,rule_original,rule_variant_original,rule_token,selected_rule_name,rule_for_refit,limit_source,selection_split,selection_strategy,grid_config_id,kept_after_within_same_model_preprocessing,kept_after_within_same_family_components,kept_after_family_level_pareto,selected_config_id,three_way_lower_threshold,three_way_upper_threshold,val3_target_miss_rate,val3_false_accept_rate,val3_uncertain_rate,val3_coverage_rate,score_conservative_target,score_balanced_reference,score_specificity_control,validation_n,validation_tp,validation_fn,validation_fp,validation_tn,validation_balanced_accuracy,validation_target_sensitivity,validation_non_target_specificity,validation_fn_rate,validation_fp_rate,validation_f1_score,validation_accuracy,validation_precision,n_projected_objects,n_projected_pixels,validation_3way_target_miss_rate,validation_3way_screening_sensitivity,validation_3way_non_target_false_accept_rate,validation_3way_uncertain_rate,validation_3way_coverage_rate,validation_3way_non_target_auto_reject_rate,validation_3way_decided_balanced_accuracy,n_components_y,n_runs,n_random_states,mean_balanced_accuracy,std_balanced_accuracy,min_balanced_accuracy,max_balanced_accuracy,mean_target_sensitivity,std_target_sensitivity,min_target_sensitivity,max_target_sensitivity,mean_non_target_specificity,std_non_target_specificity,min_non_target_specificity,max_non_target_specificity,mean_fn_rate,std_fn_rate,min_fn_rate,max_fn_rate,mean_fp_rate,std_fp_rate,min_fp_rate,max_fp_rate,threeway_n_runs,threeway_n_random_states,threeway_mean_target_miss_rate,threeway_std_target_miss_rate,threeway_min_target_miss_rate,threeway_max_target_miss_rate,threeway_mean_screening_sensitivity,threeway_std_screening_sensitivity,threeway_min_screening_sensitivity,threeway_max_screening_sensitivity,threeway_mean_non_target_false_accept_rate,threeway_std_non_target_false_accept_rate,threeway_min_non_target_false_accept_rate,threeway_max_non_target_false_accept_rate,threeway_mean_uncertain_rate,threeway_std_uncertain_rate,threeway_min_uncertain_rate,threeway_max_uncertain_rate,threeway_mean_coverage_rate,threeway_std_coverage_rate,threeway_min_coverage_rate,threeway_max_coverage_rate,threeway_mean_non_target_auto_reject_rate,threeway_std_non_target_auto_reject_rate,threeway_min_non_target_auto_reject_rate,threeway_max_non_target_auto_reject_rate,threeway_mean_decided_balanced_accuracy,threeway_std_decided_balanced_accuracy,threeway_min_decided_balanced_accuracy,threeway_max_decided_balanced_accuracy,fn_rate_border_bw3,fn_rate_core_bw3,fp_rate_border_bw3,fp_rate_core_bw3,n_errors_border_bw3,n_errors_core_bw3,n_pixels_border_bw3,n_pixels_core_bw3
0,peanut,almond,108,53,0,52,3,1.000000,0.054545,0.527273,0.518519,0.504762,0.670886,0.000000,0.945455,0.75,-0.901540,grid_empirical_cv_rules,empirical_cv_rule,object_matrix,object_mean,object_mean,40,40,not_applicable,random,absorbance_sg_smooth,absorbance+sg_smooth,data_driven_emp_cv,data_driven,11,0.01,almond,11,2,3,5,98,98,66.668457,5.685526e-05,16.527746,17.928278,1.512749,1333.969360,0.010204,0.989796,0.01,0.000204,1333.969360,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,validation_batch_3,04A_grid_rule_variant_universe,grid_001689,True,True,True,04A_object_

In [19]:
plot_detection_pareto(
    robustness_summary_df,
    fn_col="mean_fn_rate",
    fp_col="mean_fp_rate",
    color_col="mean_balanced_accuracy",
    symbol_col="matrix_family",
    id_col="selected_config_id",
    group_col="matrix_family",
    title="Robustness — mean FN/FP trade-off",
    show=True,
)

In [20]:
plot_detection_pareto(
    robustness_summary_df,
    fn_col="mean_fn_rate",
    fp_col="mean_fp_rate",
    color_col="mean_balanced_accuracy",
    symbol_col="matrix_family",
    id_col="selected_config_id",
    group_col="matrix_family",
    title="Robustness — mean FN/FP trade-off",
    show=True,
)

# 1. Selection

In [21]:
# ---------------------------------------------------------------------
# Robust Pareto selection
# ---------------------------------------------------------------------

robust_input_df = robustness_summary_df.copy()

# Optional permissive filters, not score-based.
eligible_df = robust_input_df[
    (pd.to_numeric(robust_input_df["max_fn_rate"], errors="coerce").fillna(1.0) <= MAX_ALLOWED_MAX_FN_RATE)
    & (pd.to_numeric(robust_input_df["mean_fn_rate"], errors="coerce").fillna(1.0) <= MAX_ALLOWED_MEAN_FN_RATE)
].copy()

if eligible_df.empty:
    print("[WARNING] No candidate passed permissive 2-way filters. Falling back to full robustness summary.")
    eligible_df = robust_input_df.copy()

robust_2way_pareto_df = pareto_front_by_group(
    eligible_df,
    group_cols=["matrix_family"],
    minimize_cols=[
        col for col in ROBUST_2WAY_MINIMIZE_COLS
        if col in eligible_df.columns
    ],
    maximize_cols=[
        col for col in ROBUST_2WAY_MAXIMIZE_COLS
        if col in eligible_df.columns
    ],
)

# 3-way Pareto, with a separate permissive filter.
eligible_3way_df = robust_input_df.copy()

if "threeway_mean_target_miss_rate" in eligible_3way_df.columns:
    eligible_3way_df = eligible_3way_df[
        pd.to_numeric(
            eligible_3way_df["threeway_mean_target_miss_rate"],
            errors="coerce",
        ).fillna(1.0) <= MAX_ALLOWED_MEAN_TARGET_MISS_RATE
    ].copy()

if "threeway_mean_uncertain_rate" in eligible_3way_df.columns:
    eligible_3way_df = eligible_3way_df[
        pd.to_numeric(
            eligible_3way_df["threeway_mean_uncertain_rate"],
            errors="coerce",
        ).fillna(1.0) <= MAX_ALLOWED_MEAN_UNCERTAIN_RATE
    ].copy()

if eligible_3way_df.empty:
    print("[WARNING] No candidate passed permissive 3-way filters. Falling back to full robustness summary.")
    eligible_3way_df = robust_input_df.copy()

robust_3way_pareto_df = pareto_front_by_group(
    eligible_3way_df,
    group_cols=["matrix_family"],
    minimize_cols=[
        col for col in ROBUST_3WAY_MINIMIZE_COLS
        if col in eligible_3way_df.columns
    ],
    maximize_cols=[
        col for col in ROBUST_3WAY_MAXIMIZE_COLS
        if col in eligible_3way_df.columns
    ],
)

robust_2way_ids = set(robust_2way_pareto_df["selected_config_id"].astype(str))
robust_3way_ids = set(robust_3way_pareto_df["selected_config_id"].astype(str))

robust_pareto_union_df = (
    pd.concat(
        [robust_2way_pareto_df, robust_3way_pareto_df],
        ignore_index=True,
        sort=False,
    )
    .drop_duplicates(subset=["selected_config_id"])
    .reset_index(drop=True)
)

robust_pareto_union_df["is_robust_2way_pareto"] = (
    robust_pareto_union_df["selected_config_id"].astype(str).isin(robust_2way_ids)
)

robust_pareto_union_df["is_robust_3way_pareto"] = (
    robust_pareto_union_df["selected_config_id"].astype(str).isin(robust_3way_ids)
)

robust_pareto_union_df["robust_pareto_axis"] = np.select(
    [
        robust_pareto_union_df["is_robust_2way_pareto"]
        & robust_pareto_union_df["is_robust_3way_pareto"],
        robust_pareto_union_df["is_robust_2way_pareto"],
        robust_pareto_union_df["is_robust_3way_pareto"],
    ],
    [
        "2way+3way",
        "2way",
        "3way",
    ],
    default="unknown",
)

save_parquet(robust_2way_pareto_df, ROBUST_2WAY_PARETO_PATH)
save_parquet(robust_3way_pareto_df, ROBUST_3WAY_PARETO_PATH)
save_parquet(robust_pareto_union_df, ROBUST_PARETO_UNION_PATH)

print("Robust 2-way Pareto:", robust_2way_pareto_df.shape)
print("Robust 3-way Pareto:", robust_3way_pareto_df.shape)
print("Robust union:", robust_pareto_union_df.shape)

Robust 2-way Pareto: (12, 155)
Robust 3-way Pareto: (10, 155)
Robust union: (18, 158)


In [22]:
# ---------------------------------------------------------------------
# Final panel, separated by matrix family
# ---------------------------------------------------------------------

sort_cols = [
    "matrix_family",
    "max_fn_rate",
    "mean_fn_rate",
    "std_fn_rate",
    "mean_fp_rate",
    "threeway_mean_target_miss_rate",
    "threeway_mean_uncertain_rate",
    "mean_balanced_accuracy",
]

sort_cols = [col for col in sort_cols if col in robust_pareto_union_df.columns]

ascending = []
for col in sort_cols:
    if col in {"matrix_family"}:
        ascending.append(True)
    elif col in {"mean_balanced_accuracy", "threeway_mean_coverage_rate"}:
        ascending.append(False)
    else:
        ascending.append(True)

robust_candidate_configs_df = (
    robust_pareto_union_df
    .sort_values(sort_cols, ascending=ascending)
    .groupby("matrix_family", group_keys=False, dropna=False)
    .head(N_ROBUST_PER_MATRIX_FAMILY)
    .reset_index(drop=True)
)

robust_candidate_configs_df["robust_selection_rank"] = (
    robust_candidate_configs_df
    .groupby("matrix_family")
    .cumcount()
    + 1
)

robust_candidate_configs_df["selection_strategy"] = (
    robust_candidate_configs_df["selection_strategy"].astype(str)
    + "__04B_robust_pareto"
)

config_cols = [
    "selected_config_id",
    "robust_selection_rank",
    "selection_split",
    "selection_strategy",
    "robust_pareto_axis",

    "model_family",
    "matrix_family",
    "training_matrix_id",
    "matrix_method",
    "balanced_pixel_strategy",
    "balanced_pixel_strategy_effective",
    "m",
    "m_effective",

    "preprocessing",
    "preprocessing_steps",

    "rule",
    "rule_variant",
    "selected_rule_name",
    "rule_for_refit",
    "limit_source",

    "target_class",
    "non_target_label",

    "n_components",
    "alpha",
    "object_threshold",
    "sg_window_length",
    "sg_polyorder",
    "position_dilation_radius",

    "three_way_lower_threshold",
    "three_way_upper_threshold",

    "validation_balanced_accuracy",
    "validation_target_sensitivity",
    "validation_non_target_specificity",
    "validation_fn_rate",
    "validation_fp_rate",

    "validation_3way_target_miss_rate",
    "validation_3way_screening_sensitivity",
    "validation_3way_non_target_false_accept_rate",
    "validation_3way_uncertain_rate",
    "validation_3way_coverage_rate",

    "is_robust_2way_pareto",
    "is_robust_3way_pareto",

    "mean_fn_rate",
    "std_fn_rate",
    "max_fn_rate",
    "mean_fp_rate",
    "std_fp_rate",
    "max_fp_rate",
    "mean_balanced_accuracy",

    "threeway_mean_target_miss_rate",
    "threeway_max_target_miss_rate",
    "threeway_mean_non_target_false_accept_rate",
    "threeway_mean_uncertain_rate",
    "threeway_std_uncertain_rate",
    "threeway_mean_coverage_rate",
]

config_cols = [
    col for col in config_cols
    if col in robust_candidate_configs_df.columns
]

robust_candidate_configs_df = robust_candidate_configs_df[config_cols].copy()

save_parquet(robust_candidate_configs_df, ROBUST_CANDIDATE_CONFIGS_PATH)

print("Final robust candidate configs:", robust_candidate_configs_df.shape)
print("Saved:", ROBUST_CANDIDATE_CONFIGS_PATH)

display(robust_candidate_configs_df)

Final robust candidate configs: (18, 54)
Saved: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04B_simca_validation_robustness_non_noisy_all\04B_final_candidate_panel.parquet


,selected_config_id,robust_selection_rank,selection_split,selection_strategy,robust_pareto_axis,model_family,matrix_family,training_matrix_id,matrix_method,balanced_pixel_strategy,balanced_pixel_strategy_effective,m,m_effective,preprocessing,preprocessing_steps,rule,rule_variant,selected_rule_name,rule_for_refit,limit_source,target_class,non_target_label,alpha,object_threshold,sg_window_length,sg_polyorder,position_dilation_radius,three_way_lower_threshold,three_way_upper_threshold,validation_balanced_accuracy,validation_target_sensitivity,validation_non_target_specificity,validation_fn_rate,validation_fp_rate,validation_3way_target_miss_rate,validation_3way_screening_sensitivity,validation_3way_non_target_false_accept_rate,validation_3way_uncertain_rate,validation_3way_coverage_rate,is_robust_2way_pareto,is_robust_3way_pareto,mean_fn_rate,std_fn_rate,max_fn_rate,mean_fp_rate,std_fp_rate,max_fp_rate,mean_balanced_accuracy,threeway_mean_target_miss_rate,threeway_max_target_miss_rate,threeway_mean_non_target_false_accept_rate,threeway_mean_uncertain_rate,threeway_std_uncertain_rate,threeway_mean_coverage_rate
0,04A_object_matrix_0001,1,validation_batch_3,04A_grid_rule_variant_universe__04B_robust_pareto,2way+3way,empirical_cv_rule,object_matrix,object_mean,object_mean,not_applicable,random,40,40,absorbance_sg_smooth,absorbance+sg_smooth,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,peanut,almond,0.01,0.75,11,2,3,0.05,0.95,0.527273,1.000000,0.054545,0.000000,0.945455,0.0,1.0,0.545455,0.435185,0.564815,True,True,0.000000,0.000000,0.000000,0.945455,0.000000,0.945455,0.527273,0.000000,0.000000,0.545455,0.435185,0.000000e+00,0.564815
1,04A_object_matrix_0002,2,validation_batch_3,04A_grid_rule_variant_universe__04B_robust_pareto,2way,empirical_cv_rule,object_matrix,object_median,object_median,not_applicable,random,40,40,absorbance_sg_d2,absorbance+sg_d2,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,peanut,almond,0.01,0.70,11,2,3,0.55,0.95,0.599657,0.981132,0.218182,0.018868,0.781818,0.0,1.0,0.218182,0.712963,0.287037,True,False,0.018868,0.000000,0.018868,0.781818,0.000000,0.781818,0.599657,0.000000,0.000000,0.218182,0.712963,0.000000e+00,0.287037
2,04A_object_matrix_0003,3,validation_batch_3,04A_grid_rule_variant_universe__04B_robust_pareto,2way,empirical_cv_rule,object_matrix,object_median,object_median,not_applicable,random,40,40,absorbance_sg_d2,absorbance+sg_d2,data_driven,data_driven_emp_cv,data_driven_emp_cv,data_driven_emp_cv,empirical_cv,peanut,almond,0.01,0.75,11,2,3,0.55,0.95,0.598971,0.943396,0.254545,0.056604,0.745455,0.0,1.0,0.218182,0.712963,0.287037,True,False,0.056604,0.000000,0.056604,0.745455,0.000000,0.745455,0.598971,0.000000,0.000000,0.218182,0.712963,0.000000e+00,0.287037
3,04A_object_matrix_0004,4,validation_batch_3,04A_grid_rule_variant_universe__04B_robust_pareto,2way,empirical_cv_rule,object_matrix,object_median,object_median,not_applicable,random,40,40,absorbance_sg_d2,absorbance+sg_d2,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,peanut,almond,0.01,0.70,11,2,3,0.55,0.95,0.625901,0.924528,0.327273,0.075472,0.672727,0.0,1.0,0.109091,0.814815,0.185185,True,False,0.075472,0.000000,0.075472,0.672727,0.000000,0.672727,0.625901,0.000000,0.000000,0.109091,0.814815,0.000000e+00,0.185185
4,04A_object_matrix_0005,5,validation_batch_3,04A_grid_rule_variant_universe__04B_robust_pareto,2way,empirical_cv_rule,object_matrix,object_median,object_median,not_applicable,random,40,40,absorbance_sg_d2,absorbance+sg_d2,simple,simple_emp_cv,simple_emp_cv,simple_emp_cv,empirical_cv,peanut,almond,0.01,0.75,11,2,3,0.55,0.95,0.633962,0.867925,0.400000,0.132075,0.600000,0.0,1.0,0.109091,0.814815,0.185185,True,False,0.132075,0.000000,0.132075,0.600000,0.000000,0.600000,0.633962,0.000000,0.000000,0.109091,0.814815,0.000000e+00,0.185185
5,04A_object_matrix_0006,6,validation_batch_3,04A_grid_rule_variant_universe__04B_robust_pareto,2way,empirical_cv_rul

In [23]:
protocol_df = pd.DataFrame([
    {
        "notebook": "04B_simca_validation_robustness",
        "results_tag": RESULTS_TAG,
        "target_class": TARGET_CLASS,
        "non_target_label": NON_TARGET_LABEL,

        "train_filters": json.dumps(TRAIN_FILTERS),
        "validation_filters": json.dumps(VALIDATION_FILTERS),

        "grid_summary_path": str(GRID_SUMMARY_PATH),
        "selected_candidate_configs_path": str(SELECTED_CANDIDATE_CONFIGS_PATH),

        "validation_refit_metrics_path": str(VALIDATION_REFIT_METRICS_PATH),
        "validation_refit_objects_path": str(VALIDATION_REFIT_OBJECTS_PATH),
        "validation_refit_pixels_path": str(VALIDATION_REFIT_PIXELS_PATH),

        "validation_3way_objects_path": str(VALIDATION_3WAY_OBJECTS_PATH),
        "validation_3way_metrics_path": str(VALIDATION_3WAY_METRICS_PATH),

        "random_state_stability_path": str(RANDOM_STATE_STABILITY_PATH),
        "random_state_stability_objects_path": str(RANDOM_STATE_STABILITY_OBJECTS_PATH),
        "random_state_stability_3way_metrics_path": str(RANDOM_STATE_STABILITY_3WAY_METRICS_PATH),
        "random_state_stability_3way_summary_path": str(RANDOM_STATE_STABILITY_3WAY_SUMMARY_PATH),

        "robustness_summary_path": str(ROBUSTNESS_SUMMARY_PATH),
        "robust_2way_pareto_path": str(ROBUST_2WAY_PARETO_PATH),
        "robust_3way_pareto_path": str(ROBUST_3WAY_PARETO_PATH),
        "robust_pareto_union_path": str(ROBUST_PARETO_UNION_PATH),
        "robust_candidate_configs_path": str(ROBUST_CANDIDATE_CONFIGS_PATH),

        "n_grid_summary": int(len(grid_summary_df)),
        "n_selected_configs_04a": int(len(selected_configs_df)),
        "n_candidate_panel_input": int(len(candidate_panel_df)),
        "n_validation_refit_metrics": int(len(validation_refit_metrics_df)),
        "n_validation_3way_metrics": int(len(validation_3way_metrics_df)),
        "n_random_state_stability_metrics": int(len(random_state_stability_metrics_df)),
        "n_random_state_stability_3way_metrics": int(len(random_state_stability_3way_metrics_df)),
        "n_robust_2way_pareto": int(len(robust_2way_pareto_df)),
        "n_robust_3way_pareto": int(len(robust_3way_pareto_df)),
        "n_robust_pareto_union": int(len(robust_pareto_union_df)),
        "n_final_robust_candidates": int(len(robust_candidate_configs_df)),
    }
])

save_parquet(protocol_df, VALIDATION_ROBUSTNESS_PROTOCOL_PATH)
display(protocol_df)

,notebook,results_tag,target_class,non_target_label,train_filters,validation_filters,grid_summary_path,selected_candidate_configs_path,validation_refit_metrics_path,validation_refit_objects_path,validation_refit_pixels_path,validation_3way_objects_path,validation_3way_metrics_path,random_state_stability_path,random_state_stability_objects_path,random_state_stability_3way_metrics_path,random_state_stability_3way_summary_path,robustness_summary_path,robust_2way_pareto_path,robust_3way_pareto_path,robust_pareto_union_path,robust_candidate_configs_path,n_grid_summary,n_selected_configs_04a,n_candidate_panel_input,n_validation_refit_metrics,n_validation_3way_metrics,n_random_state_stability_metrics,n_random_state_stability_3way_metrics,n_robust_2way_pareto,n_robust_3way_pareto,n_robust_pareto_union,n_final_robust_candidates
0,04B_simca_validation_robustness,non_noisy_all,peanut,almond,"{""sample_kind"": [""pure""], ""object_nut_type"": [...","{""sample_kind"": [""pure""], ""object_nut_type"": [...",C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,C:\Users\alixg\OneDrive - Université Paris-Dau...,42120,22,22,22,22,60,60,12,10,18,18
